# ML Workflow — Exercises

Companion to the note [ML Workflow](ML%20Workflow.md).

Practise the nine steps of the workflow: framing a problem with a metric and baseline, splitting before you look, leak-free preprocessing, baselines, error analysis on slices and confusion matrices, evaluating once on test, and monitoring for drift after deployment.

**15 exercises** · 🧠 Concept ×4 · ✍️ By hand ×5 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from sklearn.datasets import load_breast_cancer, load_digits
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import confusion_matrix, accuracy_score
from scipy import stats
import pandas as pd
rng = np.random.default_rng(0)

## Part A · Concepts

### Exercise 1 · Put the steps in order
*🧠 Concept · ★☆☆*

A teammate proposes this plan. Reorder it into a sound workflow and point out the two steps that are in a dangerous position.

(a) Standardise all features using the full dataset. (b) Train a gradient-boosted model. (c) Split into train/val/test.
(d) Define the metric. (e) Report test accuracy, then tune hyperparameters to improve it. (f) Fit a dummy and a logistic baseline.
(g) Look at the confusion matrix on validation data.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Which steps use information that should be unknown at training time? When may the test set be touched?

</details>

<details>
<summary><b>✅ Solution</b></summary>

Sound order: (d) metric → (c) split → (a) standardise using **train statistics only** → (f) baselines → (b) iterate models → (g) error
analysis on validation → evaluate once on test (no further tuning).

Dangerous: **(a) before (c)** leaks test statistics into training ([[Data Preprocessing]]); **(e)** tunes on the test set, so the reported
number is optimistically biased and no longer an estimate of generalization ([[Cross-Validation and Model Selection]]).

</details>

### Exercise 2 · Frame the problem
*🧠 Concept · ★★☆*

A hospital asks you to "use AI to reduce readmissions". Frame it as step 1 demands: define the **task** (input, output, prediction time),
a **metric**, a **baseline**, and two **constraints**. What would make this problem ill-posed for ML?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Be concrete: what is predicted, for whom, at what moment, and what action follows from a prediction?

</details>

<details>
<summary><b>✅ Solution</b></summary>

- **Task**: at discharge, predict whether a patient is readmitted within 30 days (binary classification) from data available at discharge only.
- **Metric**: recall at a fixed alert budget (e.g. top 10 % flagged), or PR-AUC since positives are rare; the business metric is avoided readmissions.
- **Baseline**: current clinical risk score (e.g. LACE), plus a majority-class dummy and a logistic regression.
- **Constraints**: interpretability for clinicians, privacy/regulation, latency at discharge, fairness across patient groups ([[Responsible AI]] topics).

Ill-posed if there is no action that changes the outcome, if labels are unreliable, or if the needed features are only known *after* discharge (leakage).

</details>

### Exercise 3 · Where is the leak?
*🧠 Concept · ★★☆*

Each pipeline reports suspiciously good validation results. Identify the leak.

1. Churn prediction where one feature is "number of support calls in the month **after** the prediction date".
2. Medical images: several scans per patient, randomly split into train and validation.
3. Stock-price model evaluated with shuffled k-fold CV.
4. Missing values imputed with the column mean before cross-validation.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Leakage = information at training/validation time that would not be available, or would not be independent, at real prediction time.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **Target / future leakage**: the feature is not available at prediction time and is caused by the outcome.
2. **Group leakage**: the same patient appears in train and val, so the model recognises patients. Use group k-fold.
3. **Temporal leakage**: training on the future to predict the past. Use a time-series split.
4. **Preprocessing leakage**: validation rows influence the imputer. Fit the imputer inside each fold (pipeline). Usually a small effect for a mean, but large for target encoding or feature selection.

</details>

### Exercise 4 · Data drift vs. concept drift
*🧠 Concept · ★★☆*

After deployment (step 9), explain the difference between **data (covariate) drift**, a change in $p(x)$, and **concept drift**, a change
in $p(y\mid x)$. Give an example of each for a fraud detector, and say which one you can detect *without* labels.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Labels in production often arrive late (fraud is confirmed weeks later).

</details>

<details>
<summary><b>✅ Solution</b></summary>

- **Data drift**: inputs change, e.g. a new payment app shifts the distribution of transaction amounts and times. Detectable without labels
  by comparing feature distributions (KS test, PSI) between training and production.
- **Concept drift**: the relation changes, e.g. fraudsters adopt a new pattern so the same $x$ now means fraud. Only visible with labels
  (performance monitoring) or proxies; requires retraining.

</details>

## Part B · By hand

### Exercise 5 · Is 96 % accuracy any good?
*✍️ By hand · ★☆☆*

A test set has 950 negatives and 50 positives. (a) What accuracy does the majority-class baseline get? (b) Your model gets 96 %.
By what fraction does it reduce the baseline's **error rate**? (c) What extra information would you need to judge the model?

In [ ]:
acc_baseline = None
rel_error_reduction = None

check('baseline accuracy', acc_baseline, DummyClassifier().fit(np.zeros((1000,1)), np.r_[np.zeros(950), np.ones(50)]).score(np.zeros((1000,1)), np.r_[np.zeros(950), np.ones(50)]))
check('relative error reduction', rel_error_reduction, (0.05 - 0.04) / 0.05)

<details>
<summary><b>💡 Hint</b></summary>

Error rates: baseline $0.05$, model $0.04$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $950/1000 = 0.95$. (b) $(0.05-0.04)/0.05 = 0.2$, i.e. 20 % fewer errors. (c) The confusion matrix: are any of the 50 positives found?
Precision and recall on the positive class ([[Model Evaluation and Metrics]]).

```python
acc_baseline = 0.95
rel_error_reduction = 0.2
```

</details>

### Exercise 6 · The regression baseline
*✍️ By hand · ★☆☆*

For targets $y = (2, 4, 4, 4, 5, 5, 7, 9)$ the dummy regressor predicts $\bar y$ for every sample.
(a) Compute its MSE. (b) Show that its $R^2$ is always 0. (c) A model reaches MSE $= 1$ on the same data: what is its $R^2$?

In [ ]:
mse_dummy = None
r2_model = None

_yb = np.array([2,4,4,4,5,5,7,9.])
check('dummy MSE', mse_dummy, np.var(_yb))
check('model R^2', r2_model, 1 - 1.0 / np.var(_yb))

<details>
<summary><b>💡 Hint</b></summary>

$R^2 = 1 - \frac{\text{MSE}_{\text{model}}}{\text{MSE}_{\text{mean predictor}}}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\bar y = 5$, squared deviations $(9,1,1,1,0,0,4,16)$, MSE $= 32/8 = 4$ (the population variance).
(b) $R^2 = 1 - SS_{res}/SS_{tot}$ and for the mean predictor $SS_{res}=SS_{tot}$. (c) $R^2 = 1 - 1/4 = 0.75$.

```python
mse_dummy = 4.0
r2_model = 0.75
```

</details>

### Exercise 7 · How precise is a test accuracy?
*✍️ By hand · ★★☆*

A model gets 90 % accuracy on a test set of $n=500$ i.i.d. examples. Using the normal approximation to the binomial,
give the half-width of a 95 % confidence interval. How large must $n$ be to get the half-width down to 1 percentage point?

In [ ]:
half_width = None
n_needed = None   # smallest integer

check('half-width', half_width, stats.norm.ppf(0.975) * np.sqrt(0.9*0.1/500), tol=1e-3)
check('n needed', n_needed, int(np.ceil((stats.norm.ppf(0.975)/0.01)**2 * 0.09)), tol=2)

<details>
<summary><b>💡 Hint</b></summary>

Standard error $\sqrt{\hat p(1-\hat p)/n}$; half-width $\approx 1.96\cdot$SE.

</details>

<details>
<summary><b>✅ Solution</b></summary>

SE $= \sqrt{0.09/500} = 0.0134$, half-width $= 1.96\cdot0.0134 \approx 0.026$ (so 90 % ± 2.6 points).
For $0.01$: $n = (1.96/0.01)^2\cdot0.09 \approx 3457.4$, so $n = 3458$. Small test sets give noisy estimates: differences of 1–2 points between
models are often not significant.

```python
half_width = 1.96 * np.sqrt(0.09 / 500)
n_needed = 3458
```

</details>

### Exercise 8 · Why 'evaluate once on test'
*✍️ By hand · ★★★*

You try 20 different models that **all** have true accuracy 0.80 and evaluate each on the same test set of $n=200$.
Treat the 20 test accuracies as independent $\mathcal N(0.8, \mathrm{SE}^2)$ estimates.
(a) What is the probability that a single model *appears* to reach $\ge 0.85$? (b) That at least one of the 20 does?
What does this say about picking the best model on the test set?

In [ ]:
p_single = None
p_any = None

_se = np.sqrt(0.8*0.2/200)
check('single', p_single, stats.norm.sf(0.05/_se), tol=1e-3)
check('any of 20', p_any, 1 - stats.norm.cdf(0.05/_se)**20, tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

SE $= \sqrt{0.8\cdot0.2/200}\approx0.0283$, so $0.85$ is $z\approx1.77$ standard errors above the mean.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$P(\text{at least one}) = 1 - (1 - p)^{20}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $z = 0.05/0.0283 = 1.77$, $p = 1-\Phi(1.77)\approx 0.039$. (b) $1 - (1-0.039)^{20} \approx 0.54$.
More than half the time the "winner" looks 5 points better than it is. Selecting on the test set turns it into a validation set;
the selected score is optimistically biased (the winner's curse). Select on validation / CV, and touch test once.

```python
p_single = stats.norm.sf(0.05 / np.sqrt(0.8*0.2/200))
p_any = 1 - (1 - p_single) ** 20
```

</details>

### Exercise 9 · Framing with costs: the decision threshold
*✍️ By hand · ★★☆*

A missed fraud (FN) costs 100 €, a false alarm (FP) costs 5 € of manual review, correct decisions cost 0.
Given a calibrated probability $p = P(\text{fraud}\mid x)$, derive the threshold $t$ above which flagging minimises expected cost.

In [ ]:
t_opt = None

check('threshold', t_opt, 5 / (5 + 100))

<details>
<summary><b>💡 Hint</b></summary>

Expected cost of flagging: $(1-p)\cdot5$. Expected cost of not flagging: $p\cdot100$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Flag when $(1-p)\,c_{FP} < p\,c_{FN} \iff p > \dfrac{c_{FP}}{c_{FP}+c_{FN}} = \dfrac{5}{105}\approx0.048$.
The default 0.5 threshold is only optimal for equal costs: framing the costs (step 1) directly determines the operating point.

```python
t_opt = 5 / 105
```

</details>

## Part C · Code

### Exercise 10 · Baselines first
*💻 Code · ★☆☆*

On `load_breast_cancer`, after a stratified 75/25 split (`random_state=0`), implement the majority-class baseline **by hand**
(`majority_predict(y_train, n)`) and compute its test accuracy `acc_major`. Then fit a scaled logistic regression and store its
test accuracy in `acc_logreg`.

In [ ]:
X_bc, y_bc = load_breast_cancer(return_X_y=True)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(X_bc, y_bc, test_size=0.25, random_state=0, stratify=y_bc)

def majority_predict(y_train, n):
    return None

acc_major = None
acc_logreg = None

check('majority baseline', acc_major, DummyClassifier(strategy='most_frequent').fit(Xb_tr, yb_tr).score(Xb_te, yb_te))
check('logistic regression', acc_logreg, make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xb_tr, yb_tr).score(Xb_te, yb_te))

<details>
<summary><b>💡 Hint</b></summary>

`np.bincount(y_train).argmax()` gives the majority class.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Baseline ≈ 0.63, logistic regression ≈ 0.96. A baseline gives the scale: "96 %" means removing about 90 % of the baseline's errors.
Only once a simple model is in place is it worth trying [[Ensemble Methods]] or [[Neural Networks]].

```python
X_bc, y_bc = load_breast_cancer(return_X_y=True)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(X_bc, y_bc, test_size=0.25, random_state=0, stratify=y_bc)

def majority_predict(y_train, n):
    return np.full(n, np.bincount(y_train).argmax())

acc_major = np.mean(majority_predict(yb_tr, len(yb_te)) == yb_te)
acc_logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xb_tr, yb_tr).score(Xb_te, yb_te)
print(acc_major, acc_logreg)
```

</details>

### Exercise 11 · Scaling without a leak
*💻 Code · ★★☆*

Without using `StandardScaler`, standardise `Xb_tr`/`Xb_te` correctly (statistics from train only), fit
`LogisticRegression(max_iter=1000)`, and store the test accuracy in `acc_manual`. Also store the means you used in `mu_used`.

In [ ]:
mu_used = None
acc_manual = None

check('means are train means', mu_used, Xb_tr.mean(0))
check('same as a pipeline', acc_manual, make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xb_tr, yb_tr).score(Xb_te, yb_te))

<details>
<summary><b>💡 Hint</b></summary>

`mu, sd = Xb_tr.mean(0), Xb_tr.std(0)`; apply the same `(X - mu) / sd` to both splits.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Test data must be transformed exactly as a future, unseen example would be: with the frozen training statistics. A `Pipeline` enforces this
automatically and also inside cross-validation.

```python
mu_used, sd_used = Xb_tr.mean(0), Xb_tr.std(0)
_m = LogisticRegression(max_iter=1000).fit((Xb_tr - mu_used) / sd_used, yb_tr)
acc_manual = _m.score((Xb_te - mu_used) / sd_used, yb_te)
```

</details>

### Exercise 12 · A leak that fakes success
*💻 Code · ★★★*

Create **pure noise**: `X_noise` of shape (60, 2000) and random labels `y_noise`. Any honest estimate of accuracy must be ≈ 0.5.

(a) *Leaky*: select the 20 features most correlated with $y$ (`SelectKBest(f_classif, k=20)`) on **all** data, then 5-fold CV of
logistic regression on the selected features → `acc_leaky`.
(b) *Correct*: put the selection inside a pipeline and cross-validate the pipeline → `acc_clean`.

In [ ]:
_rn = np.random.default_rng(1)
X_noise = _rn.normal(size=(60, 2000))
y_noise = _rn.integers(0, 2, 60)
acc_leaky = None
acc_clean = None

if acc_leaky is not None:
    check('leaky estimate is wildly optimistic', acc_leaky > 0.75, True)
    check('clean estimate is near chance', acc_clean < 0.7, True)
else:
    check('leaky vs clean', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression())` and `cross_val_score(pipe, X, y, cv=5).mean()`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With 2000 noise features, some correlate with the 60 labels by chance. Selecting them using all labels lets the validation folds influence
the features, and CV reports ≈ 0.9 on noise. Inside the pipeline the selection is redone on each training fold and the estimate
drops to chance. This is the textbook leak from Hastie et al. (ESL §7.10.2).

```python
_rn = np.random.default_rng(1)
X_noise = _rn.normal(size=(60, 2000))
y_noise = _rn.integers(0, 2, 60)
X_sel = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
acc_leaky = cross_val_score(LogisticRegression(), X_sel, y_noise, cv=5).mean()
acc_clean = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression()), X_noise, y_noise, cv=5).mean()
print(acc_leaky, acc_clean)
```

</details>

### Exercise 13 · Error analysis: the confusion matrix
*💻 Code · ★★☆*

Train logistic regression on a digits split. Implement `confusion(y_true, y_pred, K)` from scratch (rows = true, columns = predicted)
and find the most frequent **off-diagonal** cell as a tuple `worst_pair = (true, predicted)`. Look at a few of those images: what would you
try next?

In [ ]:
X_dg, y_dg = load_digits(return_X_y=True)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(X_dg / 16, y_dg, test_size=0.4, random_state=3, stratify=y_dg)
pred_dg = LogisticRegression(max_iter=3000).fit(Xd_tr, yd_tr).predict(Xd_te)

def confusion(y_true, y_pred, K):
    return None

cm_dg = confusion(yd_te, pred_dg, 10)
worst_pair = None

_cm = confusion_matrix(yd_te, pred_dg)
check('confusion matrix', cm_dg, _cm)
_off = _cm - np.diag(np.diag(_cm))
check('worst pair', worst_pair, np.unravel_index(_off.argmax(), _off.shape))

<details>
<summary><b>💡 Hint</b></summary>

`np.add.at(M, (y_true, y_pred), 1)`, then zero the diagonal and use `np.unravel_index(M.argmax(), M.shape)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The matrix shows *which* errors happen, not just how many. Next steps from error analysis: inspect the misclassified images, check for
label noise, add features or augmentations that address that specific confusion, or collect more data for those classes.

```python
X_dg, y_dg = load_digits(return_X_y=True)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(X_dg / 16, y_dg, test_size=0.4, random_state=3, stratify=y_dg)
pred_dg = LogisticRegression(max_iter=3000).fit(Xd_tr, yd_tr).predict(Xd_te)

def confusion(y_true, y_pred, K):
    M = np.zeros((K, K), dtype=int)
    np.add.at(M, (y_true, y_pred), 1)
    return M

cm_dg = confusion(yd_te, pred_dg, 10)
_off = cm_dg - np.diag(np.diag(cm_dg))
worst_pair = tuple(int(i) for i in np.unravel_index(_off.argmax(), _off.shape))
print(cm_dg); print('most frequent error (true, pred):', worst_pair)
```

</details>

### Exercise 14 · Error analysis on slices
*💻 Code · ★★☆*

A model can look fine on average and fail on a subgroup. The synthetic data below has a categorical `region` column.
Train logistic regression on `x1, x2`, compute test accuracy **per region** in a dict `acc_by_region`, and store the worst region in `worst_region`.

In [ ]:
n = 1500
df = pd.DataFrame({'x1': rng.normal(size=n), 'x2': rng.normal(size=n),
                   'region': rng.choice(['north', 'south', 'east'], size=n)})
logit = 2 * df.x1 - df.x2
flip = (df.region == 'east') & (rng.random(n) < 0.3)          # noisy labels in one region
df['y'] = ((logit > 0) ^ flip).astype(int)
tr, te = train_test_split(df, test_size=0.4, random_state=0)
model_s = LogisticRegression().fit(tr[['x1', 'x2']], tr.y)
acc_by_region = None
worst_region = None

_te = te.assign(ok=(model_s.predict(te[['x1', 'x2']]) == te.y))
_ref = _te.groupby('region').ok.mean()
check('per-region accuracy', None if acc_by_region is None else [acc_by_region[k] for k in _ref.index], _ref.values)
check('worst region', worst_region, _ref.idxmin())

<details>
<summary><b>💡 Hint</b></summary>

Boolean mask per region: `te.region == r`, then `np.mean(pred[mask] == te.y[mask])`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

North and south are ≈ 0.99, east only ≈ 0.72: the overall accuracy hides a broken slice. In practice slices are e.g. device types, languages
or demographic groups; slice metrics are part of error analysis and of fairness audits.

```python
n = 1500
df = pd.DataFrame({'x1': rng.normal(size=n), 'x2': rng.normal(size=n),
                   'region': rng.choice(['north', 'south', 'east'], size=n)})
logit = 2 * df.x1 - df.x2
flip = (df.region == 'east') & (rng.random(n) < 0.3)
df['y'] = ((logit > 0) ^ flip).astype(int)
tr, te = train_test_split(df, test_size=0.4, random_state=0)
model_s = LogisticRegression().fit(tr[['x1', 'x2']], tr.y)
pred_s = model_s.predict(te[['x1', 'x2']])
acc_by_region = {r: float(np.mean(pred_s[(te.region == r).values] == te.y[te.region == r])) for r in te.region.unique()}
worst_region = min(acc_by_region, key=acc_by_region.get)
print(acc_by_region)
```

</details>

### Exercise 15 · Monitoring: a drift test from scratch
*💻 Code · ★★☆*

Implement the two-sample Kolmogorov–Smirnov statistic $D = \sup_x |F_{\text{train}}(x) - F_{\text{prod}}(x)|$ in `ks_stat(a, b)`
using empirical CDFs. Apply it to a feature at training time vs. two production batches (one unchanged, one shifted)
and compare with `scipy.stats.ks_2samp`. Which batch should raise an alert?

In [ ]:
feat_train = rng.normal(0, 1, 2000)
batch_ok = rng.normal(0, 1, 500)
batch_shift = rng.normal(0.4, 1.2, 500)

def ks_stat(a, b):
    return None

D_ok, D_shift = ks_stat(feat_train, batch_ok), ks_stat(feat_train, batch_shift)

check('D (unchanged)', D_ok, stats.ks_2samp(feat_train, batch_ok).statistic)
check('D (shifted)', D_shift, stats.ks_2samp(feat_train, batch_shift).statistic)

<details>
<summary><b>💡 Hint</b></summary>

Evaluate both ECDFs at all pooled points: `np.searchsorted(np.sort(a), z, side='right') / len(a)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The sup is attained at one of the data points, so evaluating both ECDFs at the pooled sample is exact. `batch_shift` gives $D\approx0.2$
with a tiny p-value → alert; `batch_ok` gives a small $D$ consistent with sampling noise. With huge batches even irrelevant shifts are
"significant", so production systems threshold the effect size $D$ (or PSI), not just the p-value.

```python
feat_train = rng.normal(0, 1, 2000)
batch_ok = rng.normal(0, 1, 500)
batch_shift = rng.normal(0.4, 1.2, 500)

def ks_stat(a, b):
    z = np.concatenate([a, b])
    Fa = np.searchsorted(np.sort(a), z, side='right') / len(a)
    Fb = np.searchsorted(np.sort(b), z, side='right') / len(b)
    return np.max(np.abs(Fa - Fb))

D_ok, D_shift = ks_stat(feat_train, batch_ok), ks_stat(feat_train, batch_shift)
print(D_ok, stats.ks_2samp(feat_train, batch_ok).pvalue, D_shift, stats.ks_2samp(feat_train, batch_shift).pvalue)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [ML Workflow](ML%20Workflow.md).*